# SQL injection  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.1.a: Fix the Injectable Login

**Skill:** Mitigate Risk · **Maps to:** 5.1.A, 5.1.B · **Language:** Python 3 (pytest, sqlite3)

## The situation
The `authenticate` function below is the login check for a small app. It works for normal
users, but it builds its SQL query by **concatenating** the username and password straight
into the query text. That is the exact flaw that makes SQL injection possible: an attacker
who types SQL syntax instead of a username can change what the query *means*.

The starter code is the vulnerable version. Run the tests and you will see the injection
cases fail: inputs like `admin'--` and `' OR '1'='1` log in **without the real password**.

## Your task
Rewrite `authenticate` so that user input can never change the structure of the query. Use a
**parameterized query** (a prepared statement): pass the SQL template with `?` placeholders and
the user values as a separate tuple, and let the database driver bind them as data.

```python
cur.execute("SELECT role FROM users WHERE username = ? AND password = ?", (username, password))
```

When you do this, the injection inputs are treated as ordinary (non-existent) usernames, so
they return no row, while genuine credentials still succeed.

## Function contract
Implement `authenticate(users, username, password) -> str | None` in the code editor:
- `users` is a list of `(username, role, password)` tuples to load into an in-memory table.
- Return the matching user's `role` if (and only if) both username and password match a row.
- Return `None` for any failed or injected login.

Do **not** filter or blocklist the input, and do **not** compare the lists in Python. The whole
point is that the *database query itself* must be safe. Use `sqlite3` with `:memory:`.

A correct solution passes every test, including the injection cases. A string-concatenation
version will fail them, because the injected input logs in.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def authenticate(users: list[tuple[str, str, str]], username: str, password: str) -> str | None
```

### Input
- `users`: a list of `(username, role, password)` rows to load into an in-memory table.
- `username`, `password`: the submitted login credentials (untrusted input).

### Output
- The matching user's `role` (a string) when a row's username AND password both equal the
  submitted values.
- `None` when no row matches, including every injection attempt.

### Guarantees / rules
- Deterministic, no network. Standard library only (`sqlite3`).
- The username/password comparison must be performed **by a SQL query**, using a
  parameterized statement with `?` placeholders and the values bound separately.
- No input filtering, blocklisting, or Python-side list comparison.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).
the query is parameterized rather than built by string concatenation.


In [ ]:
# Reference solution
import sqlite3


def authenticate(users, username, password):
    """
    Parameterized login: the SQL template is fixed and the user values are bound
    separately as data, so injected SQL syntax is treated as an ordinary (and
    non-existent) username rather than as part of the query structure.
    """
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("CREATE TABLE users (username TEXT, role TEXT, password TEXT)")
    cur.executemany("INSERT INTO users VALUES (?, ?, ?)", users)

    cur.execute(
        "SELECT role FROM users WHERE username = ? AND password = ?",
        (username, password),
    )
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

USERS = [
    ("admin", "administrator", "S3cr3t-Admin-Pw"),
    ("jordan", "staff", "autumn-maple-31"),
    ("priya", "staff", "river-stone-77"),
]


# ---- legitimate logins still work ---------------------------------------
def test_valid_admin():
    assert impl.authenticate(USERS, "admin", "S3cr3t-Admin-Pw") == "administrator"


def test_valid_staff():
    assert impl.authenticate(USERS, "jordan", "autumn-maple-31") == "staff"


def test_wrong_password_rejected():
    assert impl.authenticate(USERS, "admin", "not-the-password") is None


def test_unknown_user_rejected():
    assert impl.authenticate(USERS, "nobody", "whatever") is None


# ---- injection must NOT authenticate (only parameterization passes these) ----
def test_comment_truncation_blocked():
    assert impl.authenticate(USERS, "admin'--", "anything") is None


def test_or_bypass_blocked():
    assert impl.authenticate(USERS, "' OR '1'='1", "x") is None


def test_or_bypass_with_comment_blocked():
    assert impl.authenticate(USERS, "' OR '1'='1'--", "x") is None


def test_union_blocked():
    assert impl.authenticate(USERS, "' UNION SELECT role FROM users--", "x") is None


# ---- a password field carrying an injection is also treated as data ----
def test_injection_in_password_blocked():
    assert impl.authenticate(USERS, "admin", "' OR '1'='1") is None


# ---- a legitimate but unusual username with a quote is handled literally ----
def test_literal_quote_username():
    users = USERS + [("o'brien", "staff", "pw-123")]
    assert impl.authenticate(users, "o'brien", "pw-123") == "staff"
    assert impl.authenticate(users, "o'brien", "wrong") is None


def test_empty_users():
    assert impl.authenticate([], "admin", "x") is None


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
